# 6 GHz quarter-wave resonator on 1 cm x 1 cm silicon chip

This notebook builds a Qiskit Metal starting layout for a capacitively coupled CPW quarter-wave resonator.  The chip is 10 mm x 10 mm, 500 um thick silicon.  The resonator target is near 6 GHz with a target linewidth of about 1 MHz.

Important: the linewidth target sets an external/load quality factor target.  The coupling length and coupling gap below are initial geometry guesses; final linewidth must be calibrated by HFSS or another EM solver sweep.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
from qiskit_metal import designs
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

In [ ]:
# Targets and substrate
target_freq_ghz = 6.0
target_linewidth_mhz = 1.0
eps_r_si = 11.7
chip_size_mm = 10.0
substrate_thickness_um = 500

# CPW starting point.  10/6 um is a common Qiskit Metal default near 50 ohm on thick Si.
cpw_width_um = 10
cpw_gap_um = 6

# Thick-substrate CPW approximation.  Good enough for initial layout length.
eps_eff = (eps_r_si + 1) / 2
c0 = 299_792_458
target_freq_hz = target_freq_ghz * 1e9
quarter_wave_length_m = c0 / (4 * target_freq_hz * math.sqrt(eps_eff))
quarter_wave_length_mm = quarter_wave_length_m * 1e3

target_loaded_q = target_freq_ghz * 1e3 / target_linewidth_mhz
assumed_internal_q = 1_000_000
target_external_q = 1 / (1 / target_loaded_q - 1 / assumed_internal_q)

print(f"eps_eff approx: {eps_eff:.3f}")
print(f"quarter-wave electrical length: {quarter_wave_length_mm:.3f} mm")
print(f"target loaded Q: {target_loaded_q:.0f}")
print(f"target external Q if Qi={assumed_internal_q:g}: {target_external_q:.0f}")

In [ ]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

design.chips.main.material = "silicon"
design.chips.main.size.size_x = f"{chip_size_mm}mm"
design.chips.main.size.size_y = f"{chip_size_mm}mm"
design.chips.main.size.size_z = f"-{substrate_thickness_um}um"
design.variables.cpw_width = f"{cpw_width_um}um"
design.variables.cpw_gap = f"{cpw_gap_um}um"

design.chips.main

In [ ]:
# Feedline and quarter-wave resonator geometry.
# The resonator route length is the open/coupled end to the shorted end.
resonator_length = f"{quarter_wave_length_mm:.3f}mm"

# Initial coupling guess for about MHz-scale linewidth.  Sweep these in HFSS.
coupling_length = "180um"
coupling_space = "6um"

LP_in = LaunchpadWirebond(
    design,
    "LP_in",
    options=dict(pos_x="-4.5mm", pos_y="3.6mm", orientation="0"),
)

readout_coupler = CoupledLineTee(
    design,
    "readout_coupler",
    options=dict(
        pos_x="0mm",
        pos_y="3.6mm",
        orientation="0",
        prime_width="cpw_width",
        prime_gap="cpw_gap",
        second_width="cpw_width",
        second_gap="cpw_gap",
        coupling_length=coupling_length,
        coupling_space=coupling_space,
        down_length="120um",
        open_termination=True,
    ),
)

LP_out = LaunchpadWirebond(
    design,
    "LP_out",
    options=dict(pos_x="4.5mm", pos_y="3.6mm", orientation="180"),
)

res_short = ShortToGround(
    design,
    "res_short",
    options=dict(pos_x="0mm", pos_y="-1.494mm", orientation="90", width="cpw_width"),
)

feed_in = RouteStraight(
    design,
    "feed_in",
    options=dict(
        pin_inputs=dict(
            start_pin=dict(component="LP_in", pin="tie"),
            end_pin=dict(component="readout_coupler", pin="prime_start"),
        ),
        trace_width="cpw_width",
    ),
)

feed_out = RouteStraight(
    design,
    "feed_out",
    options=dict(
        pin_inputs=dict(
            start_pin=dict(component="readout_coupler", pin="prime_end"),
            end_pin=dict(component="LP_out", pin="tie"),
        ),
        trace_width="cpw_width",
    ),
)

resonator = RouteStraight(
    design,
    "quarter_wave_resonator",
    options=dict(
        pin_inputs=dict(
            start_pin=dict(component="readout_coupler", pin="second_end"),
            end_pin=dict(component="res_short", pin="short"),
        ),
        trace_width="cpw_width",
    ),
)

print("components:", list(design.components.keys()))
print("route length requested:", resonator_length)
print("route length built:", f"{resonator.length:.3f} mm")
print("qgeometry rows:", {name: len(table) for name, table in design.qgeometry.tables.items()})

In [ ]:
# Open the native Qiskit Metal GUI when running interactively.
from qiskit_metal import MetalGUI

gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [ ]:
# Static matplotlib fallback preview.
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString, MultiPolygon, Polygon

def plot_geom(ax, geom, **kwargs):
    if isinstance(geom, Polygon):
        x, y = geom.exterior.xy
        ax.fill(x, y, **kwargs)
    elif isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            plot_geom(ax, part, **kwargs)
    elif isinstance(geom, LineString):
        x, y = geom.xy
        ax.plot(x, y, **kwargs)
    elif isinstance(geom, MultiLineString):
        for part in geom.geoms:
            plot_geom(ax, part, **kwargs)

fig, ax = plt.subplots(figsize=(8, 8))
chip_half = chip_size_mm / 2
ax.add_patch(plt.Rectangle((-chip_half, -chip_half), chip_size_mm, chip_size_mm, fill=False, linewidth=1.2))

for _, row in design.qgeometry.tables["poly"].iterrows():
    plot_geom(ax, row.geometry, color="#c9b48f", alpha=0.8)

for _, row in design.qgeometry.tables["path"].iterrows():
    color = "#1f77b4" if not row.get("subtract", False) else "#d7d7d7"
    alpha = 0.9 if not row.get("subtract", False) else 0.35
    plot_geom(ax, row.geometry, color=color, alpha=alpha, linewidth=2)

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-5.2, 5.2)
ax.set_ylim(-5.2, 5.2)
ax.grid(True, linewidth=0.3, alpha=0.4)
ax.set_title("6 GHz quarter-wave resonator on 10 mm x 10 mm Si chip")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
plt.show()

## HFSS tuning notes

- Frequency: sweep resonator route length around the calculated quarter-wave length.  A 1% length error moves the resonance by roughly 1%.
- Linewidth: target `kappa / 2pi = 1 MHz`, so `Q_loaded = f / linewidth ≈ 6000` at 6 GHz.
- Coupling: tune `coupling_length` and `coupling_space` in HFSS until external Q is close to 6000.  Larger coupling length and smaller coupling gap generally broaden the linewidth.
- Substrate: this notebook sets silicon and 500 um thickness; keep the package/airbox settings consistent in the renderer.